# 16 · Autocorrelação espacial: Moran global e LISA

**Vizinhança queen** (contiguidade por aresta ou vértice) sobre a **malha completa** do IBGE (`geobr`, 2024, `simplified=False`), 5.571 municípios brasileiros; exterior fora.
- **Ilhas**: Fernando de Noronha e Ilhabela não têm vizinho e saem da estatística (não são tratadas como zero).
- **Município sem par** (Boa Esperança do Norte) sai também da amostra do swing, para não virar zero.
- **Pseudo-p por permutação**: 999 permutações, semente fixa; teste **bicaudal**. O menor p possível é 0,002.
- **Correção para múltiplos testes**: LISA faz um teste por município (~5.500 por variável). Usamos **Benjamini-Hochberg com α = 0,05**. Só clusters significativos após BH são pintados (D-024).

**Ressalva**: autocorrelação é propriedade do mapa municipal. Um cluster não diz quem votou em quem, e pode refletir estrutura regional comum (ex.: Nordeste, Sul) e não um efeito local.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

In [ ]:
L = cp.lisa_pt_pl_swing(permutacoes=999)
print("municípios na amostra (sem ilhas, sem sem-par):", L["w_n"])
for nome in ("pt", "pl", "swing"):
    mo = L["moran_" + nome]
    print(
        f"Moran I ({nome}): I = {mo['I']:.3f}, E[I] = {mo['E_I']:.5f}, p_sim = {mo['p_sim']:.3f}, z = {mo['z_sim']:.1f}"
    )

In [ ]:
rows = []
for nome in ("pt", "pl", "swing"):
    t = L[nome]
    rows.append(
        {
            "variável": nome,
            **t.quadrante_sig.value_counts().to_dict(),
            "sig. sem correção (p<0,05)": int((t.p_perm < 0.05).sum()),
        }
    )
print(pd.DataFrame(rows).fillna(0).to_string(index=False))

In [ ]:
cores_q = {
    "alto-alto": "#b2182b",
    "baixo-baixo": "#2166ac",
    "alto-baixo": "#ef8a62",
    "baixo-alto": "#67a9cf",
}
for nome, titulo in (
    ("pt", "% PT (13) sobre válidos: clusters LISA (BH, α=0,05)"),
    ("pl", "% PL (22) sobre válidos: clusters LISA (BH, α=0,05)"),
    ("swing", "Swing PL 2022→2026: clusters LISA (BH, α=0,05)"),
):
    tab = L[nome][["id", "quadrante_sig"]]
    mapa = L["malha"].merge(tab, left_on="cd_mun_ibge", right_on="id", how="left")
    fig = g.mapa_categorico(
        mapa, "quadrante_sig", {**cores_q, "não significativo": "#ececec"}, titulo
    )
    g.salvar(fig, f"lisa_{nome}")
plt.show()